# 01 — Train Credit Scoring Model

Notebook train mô hình dự đoán `credit_score` từ dataset personal finance.

**Pipeline:** Load → EDA → Preprocess → Train (Linear/LightGBM/XGBoost) → Evaluate → SHAP → Export

**Output:** `model.pkl`, `preprocessor.pkl`, `features.json` — copy về thư mục `models/` của project.

## 0. Setup

Cell dưới cài các thư viện thiếu (chạy 1 lần trên Colab, bỏ qua nếu chạy local đã có `requirements.txt`).

In [ ]:
# Chỉ cần chạy trên Colab — local đã cài qua requirements.txt
# !pip install -q lightgbm xgboost shap

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42

### Load dataset

- Trên **Colab**: upload file bằng cell dưới hoặc mount Google Drive.
- Trên **local**: đường dẫn mặc định `../data/synthetic_personal_finance_dataset.csv`.

In [ ]:
# Option A — Local (mặc định)
DATA_PATH = Path('../data/synthetic_personal_finance_dataset.csv')

# Option B — Colab upload (uncomment khi cần)
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = Path(list(uploaded.keys())[0])

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. EDA — Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isna().sum()[df.isna().sum() > 0])
print(f'\nDuplicates: {df.duplicated().sum()}')

In [ ]:
# Phân phối target
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['credit_score'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Phân phối credit_score')
axes[0].axvline(580, color='red', linestyle='--', alpha=0.5, label='580 (Fair)')
axes[0].axvline(670, color='orange', linestyle='--', alpha=0.5, label='670 (Good)')
axes[0].axvline(740, color='green', linestyle='--', alpha=0.5, label='740 (Very Good)')
axes[0].axvline(800, color='blue', linestyle='--', alpha=0.5, label='800 (Exceptional)')
axes[0].legend()

sns.boxplot(x=df['credit_score'], ax=axes[1])
axes[1].set_title('Boxplot credit_score')
plt.tight_layout()
plt.show()

In [ ]:
# Phân phối các biến số
num_cols = ['age', 'monthly_income_usd', 'monthly_expenses_usd', 'savings_usd',
            'debt_to_income_ratio', 'savings_to_income_ratio']
df[num_cols].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation với credit_score
corr = df.select_dtypes(include=np.number).corr()['credit_score'].sort_values(ascending=False)
print(corr)

plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

In [ ]:
# Credit score theo các biến phân loại
cat_cols = ['education_level', 'employment_status', 'has_loan', 'region']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, cat_cols):
    sns.boxplot(data=df, x=col, y='credit_score', ax=ax)
    ax.tick_params(axis='x', rotation=30)
    ax.set_title(f'credit_score theo {col}')
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
# Drop cột ID và date (có thể extract feature từ date sau)
df_model = df.drop(columns=['user_id', 'record_date']).copy()

TARGET = 'credit_score'
y = df_model[TARGET]
X = df_model.drop(columns=[TARGET])

# Phân nhóm cột
ordinal_cols = ['education_level']
education_order = [['High School', 'Bachelor', 'Master', 'PhD']]  # thứ tự tăng dần

onehot_cols = ['gender', 'employment_status', 'job_title', 'loan_type', 'region', 'has_loan']
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

print(f'Numeric ({len(numeric_cols)}): {numeric_cols}')
print(f'Ordinal ({len(ordinal_cols)}): {ordinal_cols}')
print(f'OneHot ({len(onehot_cols)}): {onehot_cols}')

In [ ]:
# Kiểm tra giá trị unique của education để chắc thứ tự đúng
print('Education unique:', df['education_level'].unique())

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('ord', OrdinalEncoder(categories=education_order, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
    ],
    remainder='drop'
)

# Split trước, fit preprocessor CHỈ trên train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Train models

In [ ]:
def evaluate(model, X_te, y_te, name):
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2 = r2_score(y_te, pred)
    print(f'{name:20s} | MAE: {mae:6.2f} | RMSE: {rmse:6.2f} | R²: {r2:.4f}')
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'pred': pred}

In [ ]:
results = []

# Baseline — Linear Regression
pipe_lr = Pipeline([('pre', preprocessor), ('model', LinearRegression())])
pipe_lr.fit(X_train, y_train)
results.append(evaluate(pipe_lr, X_test, y_test, 'Linear Regression'))

In [ ]:
# LightGBM
pipe_lgbm = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        random_state=RANDOM_STATE, verbose=-1
    ))
])
pipe_lgbm.fit(X_train, y_train)
results.append(evaluate(pipe_lgbm, X_test, y_test, 'LightGBM'))

In [ ]:
# XGBoost
pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('model', XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        random_state=RANDOM_STATE, verbosity=0
    ))
])
pipe_xgb.fit(X_train, y_train)
results.append(evaluate(pipe_xgb, X_test, y_test, 'XGBoost'))

In [ ]:
# So sánh
res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'pred'} for r in results])
res_df = res_df.sort_values('rmse')
res_df

In [ ]:
# Chọn model tốt nhất (RMSE thấp nhất)
best_name = res_df.iloc[0]['model']
best_pipe = {'Linear Regression': pipe_lr, 'LightGBM': pipe_lgbm, 'XGBoost': pipe_xgb}[best_name]
print(f'Best model: {best_name}')

In [ ]:
# Predicted vs Actual + Residuals
best_pred = best_pipe.predict(X_test)
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, best_pred, alpha=0.3, s=10)
axes[0].plot([300, 850], [300, 850], 'r--')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'{best_name} — Predicted vs Actual')

axes[1].scatter(best_pred, residuals, alpha=0.3, s=10)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual plot')
plt.tight_layout()
plt.show()

## 4. SHAP — giải thích mô hình

Áp dụng SHAP trên model tốt nhất (nếu là tree-based).

In [ ]:
import shap

# Transform X_test qua preprocessor để lấy feature names cuối cùng
X_test_transformed = best_pipe.named_steps['pre'].transform(X_test)
feature_names_out = best_pipe.named_steps['pre'].get_feature_names_out()

# Explainer chỉ áp dụng cho tree-based
if best_name in ('LightGBM', 'XGBoost'):
    explainer = shap.TreeExplainer(best_pipe.named_steps['model'])
    # Chỉ lấy 500 samples cho nhanh
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test_transformed), 500, replace=False)
    shap_values = explainer.shap_values(X_test_transformed[sample_idx])

    shap.summary_plot(shap_values, X_test_transformed[sample_idx],
                      feature_names=feature_names_out, max_display=15)
else:
    print('Bỏ qua SHAP vì model không phải tree-based.')

In [ ]:
# Waterfall plot cho 1 sample
if best_name in ('LightGBM', 'XGBoost'):
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test_transformed[sample_idx][0],
        feature_names=feature_names_out.tolist()
    ), max_display=12)

## 5. Export artifacts

Xuất 3 file: `model.pkl` (toàn bộ pipeline), `preprocessor.pkl` (rời — nếu cần dùng riêng), `features.json` (schema input).

In [ ]:
OUT_DIR = Path('../models')
OUT_DIR.mkdir(exist_ok=True, parents=True)

# 1. Full pipeline (preprocessor + model) — Streamlit chỉ cần file này
joblib.dump(best_pipe, OUT_DIR / 'model.pkl')

# 2. Preprocessor rời (backup)
joblib.dump(best_pipe.named_steps['pre'], OUT_DIR / 'preprocessor.pkl')

# 3. Schema — Streamlit dùng để build form và đảm bảo đúng thứ tự cột
schema = {
    'target': TARGET,
    'best_model': best_name,
    'metrics': {k: float(v) for k, v in res_df.iloc[0].to_dict().items() if k != 'model'},
    'input_columns': list(X.columns),
    'numeric_cols': numeric_cols,
    'ordinal_cols': ordinal_cols,
    'onehot_cols': onehot_cols,
    'education_order': education_order[0],
    'categorical_values': {
        c: sorted(df[c].dropna().unique().tolist()) for c in onehot_cols
    },
    'numeric_ranges': {
        c: {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
        for c in numeric_cols
    }
}

with open(OUT_DIR / 'features.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print('Exported:')
for p in OUT_DIR.iterdir():
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## 6. Sanity check — load lại và predict thử

Đảm bảo file `.pkl` load được và cho ra kết quả đúng.

In [ ]:
loaded = joblib.load(OUT_DIR / 'model.pkl')
sample = X_test.iloc[[0]]
print('Input:')
print(sample.T)
print(f'\nActual credit_score: {y_test.iloc[0]}')
print(f'Predicted:           {loaded.predict(sample)[0]:.1f}')

## 7. (Colab) Tải file về máy

Chạy cell dưới nếu đang trên Colab — sẽ trigger download 3 file.

In [ ]:
# from google.colab import files
# for name in ['model.pkl', 'preprocessor.pkl', 'features.json']:
#     files.download(str(OUT_DIR / name))